## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-27-00-54-45 +0000,bbc,I spend £200 a month on fancy meals by myself ...,https://www.bbc.co.uk/news/articles/cw8dv11dp2...
1,2026-11-27-00-53-04 +0000,nypost,Family of Alaska mom who died after walking in...,https://nypost.com/2026/11/26/us-news/family-o...
2,2026-11-27-00-52-11 +0000,bbc,"Major Thanksgiving storm brings flooding, flig...",https://www.bbc.co.uk/news/articles/ckm2mz3prq...
3,2026-11-27-00-48-11 +0000,bbc,"Met Police ruined my life, says Kenyan man wro...",https://www.bbc.co.uk/news/articles/cqvg2nlr8k...
4,2026-11-27-00-47-43 +0000,bbc,Death-sentenced Sheikh Hasina's planned return...,https://www.bbc.co.uk/news/articles/cx30g71rev...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2655/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
19,thanksgiving,28
319,trump,19
212,he,14
57,new,14
791,china,11


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
66,2026-11-26-20-37-19 +0000,nypost,Trump celebrates Thanksgiving with his holiday...,https://nypost.com/2026/11/26/us-news/trump-ce...,73
149,2026-11-26-14-47-00 +0000,wsj,Macy’s Postpones Thanksgiving Day Parade as Po...,https://www.wsj.com/us-news/macys-postpones-it...,68
160,2026-11-26-13-58-19 +0000,nypost,Trump kicks off Thanksgiving with late night d...,https://nypost.com/2026/11/26/us-news/presiden...,65
135,2026-11-26-15-39-23 +0000,nypost,Trump tells Putin to ‘pull back all military a...,https://nypost.com/2026/11/26/us-news/trump-te...,65
240,2026-11-26-10-00-00 +0000,wapo,"Which came first, Macy’s Thanksgiving parade o...",https://www.washingtonpost.com/nation/interact...,61


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
66,73,2026-11-26-20-37-19 +0000,nypost,Trump celebrates Thanksgiving with his holiday...,https://nypost.com/2026/11/26/us-news/trump-ce...
27,50,2026-11-26-23-03-16 +0000,bbc,My dad's watch stopped the moment he died - I ...,https://www.bbc.co.uk/news/videos/c6evr2y40v0j...
219,48,2026-11-26-10-25-45 +0000,nypost,Russia targets Kyiv with ballistic missiles — ...,https://nypost.com/2026/11/26/world-news/russi...
113,47,2026-11-26-17-12-53 +0000,wapo,Two Italians killed in Kuwait as a strained Ir...,https://www.washingtonpost.com/world/2026/11/2...
149,40,2026-11-26-14-47-00 +0000,wsj,Macy’s Postpones Thanksgiving Day Parade as Po...,https://www.wsj.com/us-news/macys-postpones-it...
71,38,2026-11-26-20-30-14 +0000,nypost,Nvidia halts server shipments to 4 countries j...,https://nypost.com/2026/11/26/business/nvidia-...
258,37,2026-11-26-08-42-00 +0000,wsj,New Tariffs on China Risk Undermining Trade Tr...,https://www.wsj.com/world/new-tariffs-on-china...
237,32,2026-11-26-10-00-00 +0000,wsj,Democratic Sen. Chris Van Hollen wants to be S...,https://www.wsj.com/politics/policy/why-this-d...
306,31,2026-11-26-01-28-12 +0000,latimes,"Trump, FCC illegally sought to punish ABC in f...",https://www.latimes.com/entertainment-arts/bus...
29,30,2026-11-26-23-01-05 +0000,latimes,One flood threat eases as another looms: Heavy...,https://www.latimes.com/california/story/2026-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
